This notebook is for all my actual model building and data preprocessing. My EDA is done in this other notebook: https://www.kaggle.com/code/richardhhong/calories-eda  
Also I know that a lot of my function and variable names suck but thats not important right now ~

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter("ignore")

In [2]:
df_train = pd.read_csv("/kaggle/input/playground-series-s5e5/train.csv")
df_test = pd.read_csv("/kaggle/input/playground-series-s5e5/test.csv")

# Data Preprocessing

In [3]:
num_vars = df_train.drop(columns=['id','Calories']).select_dtypes(include=['int64', 'float64']).columns
cat_vars = ['Sex']

# Feature Engineering

In [4]:
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler

In [5]:
df_train.head()

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,4,female,38,166.0,61.0,25.0,102.0,40.6,146.0


In [6]:
# encodings for some stuff
from itertools import combinations

def bmi_to_weighttype(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif bmi <= 24.9:
        return "NormalWeight"
    elif bmi <= 29.9:
        return "Overweight"
    else:
        return "Obesity"

def age_to_group(age):
    if age <= 18:
        return "Child"
    elif age <= 30:
        return "Young Adult"
    elif age <= 50:
        return "Adult"
    else:
        return "Senior"

def make_pairs(df):
    df_temp = df.copy()
    encode_columns = ['Sex', 'WeightType', 'AgeGroup']
    pair_size = [2,3]
    
    for r in pair_size:
        for cols in list(combinations(encode_columns, r)):
            new_col_name = '_'.join(cols)
            
            df_temp[new_col_name] = df_temp[list(cols)].astype(str).agg('_'.join, axis=1)
            df_temp[new_col_name] = df_temp[new_col_name].astype('category')

    return df_temp

In [7]:
def make_other_ints(df):
    df_temp = df.copy()
    df_temp["HeartLoad"] = df_temp["Heart_Rate"] * df_temp["Duration"]
    df_temp["TempHeartInteraction"] = df_temp["Body_Temp"] * df_temp["Heart_Rate"]
    df_temp["Weight_per_Height"] = df_temp["Weight"] / df_temp["Height"]
    df_temp["HeartRate_per_Age"] = df_temp["Heart_Rate"] / df_temp["Age"]
    df_temp['Intensity'] = df_temp['Heart_Rate'] / df_temp['Duration']
    return df_temp

def make_bmr(df):
    df_temp = df.copy()
    df_temp['BMR']=0
    df_temp.loc[df_temp.Sex=='male','BMR'] = df_temp['Weight'] * 9.65 + (df_temp['Height'] / 100) * 573 - df_temp['Age'] * 5.08 + 260
    df_temp.loc[df_temp.Sex=='female','BMR'] = df_temp['Weight'] * 7.38 + (df_temp['Height'] / 100) * 607 - df_temp['Age'] * 2.31 + 43
    return df_temp

def make_features(df, test=False, make_log=True, make_int=True, use_pca=None):
    df_temp = df.copy()

    df_temp.drop(columns=['id'], inplace=True)
    
    # dummy encoding
    # df_temp = pd.get_dummies(df_temp, columns=['Sex'])
    df_temp['Sex'] = df_temp['Sex'].astype("category")

    # new features, to be culled off in feature selection
    df_temp['BMI'] = df_temp['Weight'] / (df_temp['Height']/100)**2

    df_temp["WeightType"] = df_temp["BMI"].apply(bmi_to_weighttype).astype("category")

    df_temp.drop(columns=['BMI'], inplace=True)
    
    df_temp["AgeGroup"] = df_temp["Age"].apply(age_to_group).astype("category")

    df_temp = make_bmr(df_temp)

    df_temp = make_other_ints(df_temp)
    df_temp = make_pairs(df_temp)

    return df_temp

# for predicting log of outcome rather than just outcome
def make_outcome_log(df, make_int=True, make_log=True):
    df_temp = df.copy()

    df_temp['log_calories'] = np.log1p(df_temp['Calories'])
    
    return df_temp

def make_outcome_calmin(df, make_int=True, make_log=True):
    df_temp = df.copy()
    
    df_temp['Calories_Minute']=df_temp['Calories']/df_temp['Duration']

    return df_temp

df_train1 = make_features(df_train, make_int = False, make_log=False)
df_train2 = make_outcome_log(df_train, make_int = False, make_log=False)
df_train3 = make_outcome_calmin(df_train, make_int = False, make_log = False)

In [8]:
df_train1.head()

,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories,WeightType,AgeGroup,BMR,HeartLoad,TempHeartInteraction,Weight_per_Height,HeartRate_per_Age,Intensity,Sex_WeightType,Sex_AgeGroup,WeightType_AgeGroup,Sex_WeightType_AgeGroup
0,male,36,189.0,82.0,26.0,101.0,41.0,150.0,NormalWeight,Adult,1951.39,2626.0,4141.0,0.433862,2.805556,3.884615,male_NormalWeight,male_Adult,NormalWeight_Adult,male_NormalWeight_Adult
1,female,64,163.0,60.0,8.0,85.0,39.7,34.0,NormalWeight,Senior,1327.37,680.0,3374.5,0.368098,1.328125,10.625000,female_NormalWeight,female_Senior,NormalWeight_Senior,female_NormalWeight_Senior
2,female,51,161.0,64.0,7.0,84.0,39.8,29.0,NormalWeight,Senior,1374.78,588.0,3343.2,0.397516,1.647059,12.000000,female_NormalWeight,female_Senior,NormalWeight_Senior,female_NormalWeight_Senior
3,male,20,192.0,90.0,25.0,105.0,40.7,140.0,NormalWeight,Young Adult,2127.06,2625.0,4273.5,0.468750,5.250000,4.200000,male_NormalWeight,male_Young Adult,NormalWeight_Young Adult,male_NormalWeight_Young Adult
4,female,38,166.0,61.0,25.0,102.0,40.6,146.0,NormalWeight,Adult,1413.02,2550.0,4141.2,0.367470,2.684211,4.080000,female_NormalWeight,female_Adult,NormalWeight_Adult,female_NormalWeight_Adult


# Model

## XGB Baseline

In [9]:
SEED = 30

In [10]:
import xgboost as xgb
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import train_test_split

In [11]:
# without the fancy features
X = df_train.drop(columns=['id', 'Calories'])
X['Sex'] = X['Sex'].astype('category')
y = df_train['Calories']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=SEED)

# with the fancy features
X1 = df_train1.drop(columns=['Calories'])
y1 = df_train1['Calories']
X_train1, X_val1, _, _ = train_test_split(X1, y1, test_size=0.3, random_state=SEED)

# with log calories
y2 = df_train2['log_calories']
_, _, y_train2, y_val2 = train_test_split(X, y2, test_size=0.3, random_state=SEED)

# with calories_minutes
y3 = df_train3['Calories_Minute']
_, _, y_train3, y_val3 = train_test_split(X, y3, test_size=0.3, random_state=SEED)

In [12]:
# baseline without new features
xgb_baseline = xgb.XGBRegressor(enable_categorical=True)
xgb_baseline.fit(X_train, y_train)

y_val_pred = xgb_baseline.predict(X_val)
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(f'XGB Baseline Score: {score}')

XGB Baseline Score: 0.06565689523467988


In [13]:
# baseline with new features
xgb_baseline1 = xgb.XGBRegressor(enable_categorical=True)
xgb_baseline1.fit(X_train1, y_train)

y_val_pred = xgb_baseline1.predict(X_val1)
y_val_pred = np.maximum(y_val_pred, 0)
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(f'XGB Baseline Score With new Features: {score}')

XGB Baseline Score With new Features: 0.06558647386421829


In [14]:
# baseline with log calories
xgb_baseline2 = xgb.XGBRegressor(enable_categorical=True)
xgb_baseline2.fit(X_train, y_train2)

y_val_pred = xgb_baseline2.predict(X_val)
y_val_pred = np.expm1(y_val_pred)
y_val_pred = np.maximum(y_val_pred, 0)
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(f'XGB Baseline Score With log Calories: {score}')

XGB Baseline Score With log Calories: 0.06252807500452165


In [15]:
# baseline with log calories and the new features
xgb_baseline3 = xgb.XGBRegressor(enable_categorical=True)
xgb_baseline3.fit(X_train1, y_train2)

y_val_pred = xgb_baseline3.predict(X_val1)
y_val_pred = np.expm1(y_val_pred)
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(f'XGB Baseline Score With new Features and Log Calories: {score}')

XGB Baseline Score With new Features and Log Calories: 0.062014074481138236


In [16]:
# baseline with calories_minute and the new features
xgb_baseline4 = xgb.XGBRegressor(enable_categorical=True)
xgb_baseline4.fit(X_train1, y_train3)

y_val_pred = xgb_baseline4.predict(X_val1)
y_val_pred = y_val_pred * X_val1['Duration']
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(f'XGB Baseline Score With new Features and Calories Minutes: {score}')

XGB Baseline Score With new Features and Calories Minutes: 0.06132996655464431


## Big Tuna

In [17]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

In [18]:
def rmsle_eval(y_pred, dtrain):
    y_true = dtrain.get_label()
    y_pred = np.maximum(y_pred, 0)
    loss = np.sqrt(mean_squared_log_error(y_true, y_pred))
    return 'RMSLE', loss

# Function to run k-fold cross-validation with XGBoost and MSLE
def xgb_cv_rmsle(X, y, params, num_folds=5, debug=False, log=False, y_act=y):
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)
    fold_scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        y_val_act = y_act.iloc[val_idx]
        
        model = xgb.XGBRegressor(
            **params,
            enable_categorical=True
        )
        model.fit(X_train,y_train)
        
        y_val_pred = model.predict(X_val)
        y_val_pred = np.maximum(0, y_val_pred)

        if log == True:
            y_val_pred = np.expm1(y_val_pred)
            
        score = np.sqrt(mean_squared_log_error(y_val_act, y_val_pred))

        if debug == True:
            print(score)
            
        fold_scores.append(score)
        
    return fold_scores

In [19]:
def objective(trial):
    params = {
        "objective": "reg:squarederror",
        "eval_metric" : "rmse",
        "tree_method": "gpu_hist",
        "predictor": "gpu_predictor",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, step=0.01),
        "max_depth": trial.suggest_int("max_depth", 5, 30),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0, step=0.1),
        "max_bin": trial.suggest_int("max_bin", 256, 2048),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 0.1, step=0.01),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "n_estimators": trial.suggest_int("n_estimators", 50, 2000),
        "max_delta_step": trial.suggest_int("max_delta_step", 1, 10),
        "random_state": SEED
    }

    score = np.mean(xgb_cv_rmsle(X=X1, y=y2, params=params, debug=False, log=True))
    return score

In [20]:
# %%time
# study = optuna.create_study(direction='minimize',
#                             sampler = optuna.samplers.RandomSampler(seed=SEED),
#                             study_name = "BIG BLUE FIN TUNA!!")
# study.optimize(objective, n_trials=100, show_progress_bar=True, )

In [21]:
# # best_params = study.best_params
# print(f'Best Trial Params: {best_params}')

# print(f'Best Trial Value: {study.best_trial.value}')

In [22]:
best_params = {'learning_rate': 0.03, 
               'max_depth': 20, 
               'subsample': 0.7, 
               'colsample_bytree': 0.7, 
               'max_bin': 774, 
               'min_child_weight': 4, 
               'gamma': 0.08, 
               'lambda': 1.7820438719237879, 
               'alpha': 0.3170600920490681, 
               'grow_policy': 'depthwise',
               'n_estimators': 1199, 
               'max_delta_step': 1}

# Submission

In [23]:
best_model = xgb.XGBRegressor(**best_params, enable_categorical=True)
best_model.fit(X_train1, y_train3)

XGBRegressor(alpha=0.3170600920490681, base_score=None, booster=None,
             callbacks=None, colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             gamma=0.08, grow_policy='depthwise', importance_type=None,
             interaction_constraints=None, lambda=1.7820438719237879,
             learning_rate=0.03, max_bin=774, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=1, max_depth=20,
             max_leaves=None, min_child_weight=4, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1199,
             n_jobs=None, ...)

In [24]:
y_val_pred = best_model.predict(X_val1)
y_val_pred = y_val_pred * X_val1['Duration']
score = mean_squared_log_error(y_val_pred, y_val)
print(score)
# Why the heck is this score so low????

0.0038303101676357386


In [25]:
df_test1 = make_features(df_test, test=True, make_int=False, make_log=False)

df_test1.head()

,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,WeightType,AgeGroup,BMR,HeartLoad,TempHeartInteraction,Weight_per_Height,HeartRate_per_Age,Intensity,Sex_WeightType,Sex_AgeGroup,WeightType_AgeGroup,Sex_WeightType_AgeGroup
0,male,45,177.0,81.0,7.0,87.0,39.8,Overweight,Adult,1827.26,609.0,3462.6,0.457627,1.933333,12.428571,male_Overweight,male_Adult,Overweight_Adult,male_Overweight_Adult
1,male,26,200.0,97.0,20.0,101.0,40.5,NormalWeight,Young Adult,2209.97,2020.0,4090.5,0.485000,3.884615,5.050000,male_NormalWeight,male_Young Adult,NormalWeight_Young Adult,male_NormalWeight_Young Adult
2,female,29,188.0,85.0,16.0,102.0,40.4,NormalWeight,Young Adult,1744.47,1632.0,4120.8,0.452128,3.517241,6.375000,female_NormalWeight,female_Young Adult,NormalWeight_Young Adult,female_NormalWeight_Young Adult
3,female,39,172.0,73.0,20.0,107.0,40.6,NormalWeight,Adult,1535.69,2140.0,4344.2,0.424419,2.743590,5.350000,female_NormalWeight,female_Adult,NormalWeight_Adult,female_NormalWeight_Adult
4,female,30,173.0,67.0,16.0,94.0,40.5,NormalWeight,Young Adult,1518.27,1504.0,3807.0,0.387283,3.133333,5.875000,female_NormalWeight,female_Young Adult,NormalWeight_Young Adult,female_NormalWeight_Young Adult


In [26]:
y_test_pred = best_model.predict(df_test1)
# y_test_pred = np.expm1(y_test_pred)
y_test_pred = y_test_pred * df_test1['Duration']
y_test_pred = np.maximum(y_test_pred, 0)

submission = pd.read_csv("/kaggle/input/playground-series-s5e5/sample_submission.csv")
submission['Calories'] = y_test_pred
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Calories
0,750000,27.736211
1,750001,112.607794
2,750002,87.330421
3,750003,125.009794
4,750004,75.746078
